#  TransactFlow notebook

The notebook for TransactFlow, for reading, processing and analytics of personal financial transactions.

This notebook is meant to focus on visualizing data, while most of the processing logic goes to other `.py` files in this directory.

## Preparation

Import everything:

In [ ]:
import os
import filecmp
import locale
import ipywidgets as widgets
from IPython.core.display import HTML
from IPython.display import display
from ipywidgets import interact, interact_manual
from typing import cast
import bqplot as bq
import bqplot.pyplot
from colors import COLORS, XKCD_TO_HEX

from base import *
import processes.runAll
from analysis import AnalysisProvider, AnalysisProviderOptions, SegmentedDisplayOption, LabelSetAlias, ANY_CATEGORY, DeductIncomeOption, AnalysisProviderFilter, GroupLabelOption, CategorizeOption, transListToHtmlTable

If not already, change to the correct working directory:

In [ ]:
#DIRECTORY = "/path/to/project"
#alreadyHere = filecmp.cmp(DIRECTORY, os.getcwd())
#if not alreadyHere:
#    os.chdir(DIRECTORY)
#print(f"We are at: {os.getcwd()}")
print(os.listdir())

Import libraries and other Python files, then set current locale to JP:

In [ ]:
try:
    locale.setlocale(locale.LC_ALL, "ja_JP")
    locale.getlocale()
except: pass

### Import Transactions

Load all transactions, split them into groups by salary (ignore leading, since Feb 2019 - Mar 2019 is pretty much empty):


In [ ]:
trans = processes.runAll.run(progress=False)

minDate, maxDate = minMaxDateOf(trans)
accounts = {t.account for t in trans}
print(f"Imported {len(trans)} transactions from {minDate} to {maxDate}")
print(f"...for accounts: {accounts}")
# for t in trans: print(t)

In [ ]:
provider = AnalysisProvider(trans)

print(list(enumerate(provider.labels)))

## Visualization Setup

### Utility Functions

These will come in handy later:

In [ ]:
def formatAmount(amount):
    if amount < 10000: return int(amount)
    return f"{amount/10000:.2f}万"

def formatAmountInCNY(amount, provider):
    return f"CNY {formatAmount(provider.rates.JPYCNYRate * amount)}"

### ipywidgets

Customize widget layout a bit:

In [ ]:
# This updates the description labels to the left of dropdown/text input.
display(HTML('''<style>
    .widget-label { min-width: 20ex !important; }
</style>'''))

def widerDropdown(**kwargs):
    return widgets.Dropdown(**kwargs, layout={'width': '500px'})
def widerTextInput(**kwargs):
    return widgets.Text(**kwargs, layout={'width': '500px'})
def dropdownWithEnumValuesAsLabels(enumType, **dropdownKwargs):
    widgetOptions = [ (c.value, c) for c in list(enumType) ]
    return widerDropdown(options=widgetOptions, **dropdownKwargs)


Utility function to provide shared widgets that make analysis provider options.

In [ ]:
categoryOptions = ORDERED_BASE_CATEGORIES + [
    ANY_CATEGORY
]

groupLabelWidgetOptions = [
    (label, cast(GroupLabelOption, label)) for label in provider.labels
] + [
    (alias.value, cast(GroupLabelOption, alias)) for alias in list(LabelSetAlias)
]

categoryFilterWidgetOptions = [ (cat.label, cat) for cat in categoryOptions ]

ANY_ACCOUNT = "Any account"
accountWidgetOptions = [ANY_ACCOUNT] + ORDERED_ACCOUNTS

class SharedWidgets:
    labelWidget = widerDropdown(options=groupLabelWidgetOptions,
                                value=LabelSetAlias.MONTHS_2024,
                                description="Time interval")
    categoryFilterWidget = widerDropdown(options=categoryFilterWidgetOptions,
                                         value=ANY_CATEGORY,
                                         description="Category filter")
    descriptionContainsWidget = widerTextInput(value="",
                                               description="Description contains")
    amountFromWidget = widgets.FloatText(value=-1_0000_0000, description="Amount quantity from")
    amountUntilWidget = widgets.FloatText(value=1_0000_0000, description="Amount quantity until")
    filterByRawAmountWidget = widgets.Checkbox(value=False, description="Filter by raw amount")
    recordAccountWidget = widerDropdown(options=accountWidgetOptions,
                                        value=ANY_ACCOUNT,
                                        description="Record account")
    exactMatchCategoryWidget = widgets.Checkbox(value=False, description="Exact match category")
    segmentedDisplayWidget = dropdownWithEnumValuesAsLabels(SegmentedDisplayOption,
                                                            value=SegmentedDisplayOption.NO_SPEC,
                                                            description="Segmented Display")
    categorizeOptionWidget = dropdownWithEnumValuesAsLabels(CategorizeOption,
                                                            value=CategorizeOption.ORIGINAL,
                                                            description="Categorize option")
    providerOptionsWidgetGroup = widgets.VBox([
        labelWidget,
        descriptionContainsWidget,
        amountFromWidget,
        amountUntilWidget,
        filterByRawAmountWidget,
        recordAccountWidget,
        categoryFilterWidget,
        exactMatchCategoryWidget,
        segmentedDisplayWidget,
        categorizeOptionWidget
    ])
    
def makeProviderOptionsArgsToWidgets():
    return {
        "labelOption": SharedWidgets.labelWidget,
        "categoryFilter": SharedWidgets.categoryFilterWidget,
        "descriptionContains": SharedWidgets.descriptionContainsWidget,
        "amountQuantityFrom": SharedWidgets.amountFromWidget,
        "amountQuantityUntil": SharedWidgets.amountUntilWidget,
        "filterByRawAmount": SharedWidgets.filterByRawAmountWidget,
        "recordAccount": SharedWidgets.recordAccountWidget,
        "exactMatchCategory": SharedWidgets.exactMatchCategoryWidget,
        "segmentedDisplayOption": SharedWidgets.segmentedDisplayWidget,
        "categorizeOption": SharedWidgets.categorizeOptionWidget
    }

def makeOptions(**kwargs):
    argNamesForOptions = set(makeProviderOptionsArgsToWidgets().keys())
    optionsKwargs = {}
    for argName in argNamesForOptions:
        optionsKwargs[argName] = kwargs.pop(argName)
    unusedKwargs = kwargs
    # Handle non-filter option args
    categorizeOption = optionsKwargs.pop("categorizeOption")
    filterKwargs = optionsKwargs
    # Handle special cases in filter args
    if filterKwargs["categoryFilter"] == ANY_CATEGORY:
        filterKwargs["categoryFilter"] = None
    if filterKwargs["recordAccount"] == ANY_ACCOUNT:
        filterKwargs["recordAccount"] = None
    providerFilter = AnalysisProviderFilter(**filterKwargs)
    return (AnalysisProviderOptions(providerFilter, categorizeOption),
            unusedKwargs)

def mergedDict(*dicts):
    merged = {}
    for d in dicts:
        for k, v in d.items(): merged[k] = v
    return merged

def linkWithSharedWidgets(displaySharedWidgets = True, manual = False, **kwargs):
    extraArgsToWidgets = kwargs
    extraArgs = set(extraArgsToWidgets.keys())
    optionsArgs = set(makeProviderOptionsArgsToWidgets().keys())
    assert(len(extraArgs & optionsArgs) == 0)
    def wrapper(fn):
        namesToWidgets = mergedDict(makeProviderOptionsArgsToWidgets(),
                                    extraArgsToWidgets)
        def makeOptionsAndCallFn(**kwargs):
            options, unusedKwargs = makeOptions(**kwargs)
            fn(options=options, **unusedKwargs)
        if displaySharedWidgets:
            display(SharedWidgets.providerOptionsWidgetGroup)
        if manual:
            manualUpdateButton = widgets.Button(description="Update")
            out = widgets.Output()
            def onButtonClick(btn):
                out.clear_output()
                kwargsToPass = { name: w.value for name, w in namesToWidgets.items() }
                makeOptionsAndCallFn(**kwargsToPass)
            manualUpdateButton.on_click(onButtonClick)
            display(manualUpdateButton)
        else:
            out = widgets.interactive_output(makeOptionsAndCallFn, namesToWidgets)
        for extraWidget in extraArgsToWidgets.values():
            display(extraWidget)
        display(out)
    return wrapper

SharedWidgets.providerOptionsWidgetGroup

Other shared widgets for analysis provider:

In [ ]:
includeRemainingWidget = widgets.ToggleButton(value=False, description="Include remaining")
display(includeRemainingWidget)

deductIncomeOptionWidget = dropdownWithEnumValuesAsLabels(DeductIncomeOption,
                                                          description="Deduct salary option")
display(deductIncomeOptionWidget)

averageByGroupWidget = widgets.ToggleButton(value=False, description="Average by group")
display(averageByGroupWidget)

## Visualization

### Overview

In [ ]:
# for label in provider.labelOptions:
for label in ["All", "2019", "2020", "2021", "2022", "2023", "2024"]:
    print(provider.groupOverview(label))

print(provider.netTotalsReport())

### List Transactions

In [ ]:
transListHTML = widgets.HTML()
transStatsLabel = widgets.Label()

@linkWithSharedWidgets(colorOn=widgets.Checkbox(value=True, description="color on"))
def displayTable(options, colorOn):
    stats = provider.transactionSetStatsMatching(options)
    selected = stats.transactions
    labelStringComponents = []
    if len(selected) == 0:
        labelStringComponents.append("No transactions to display.")
        return
    numItemsLimit = 600
    if len(selected) > numItemsLimit:
        labelStringComponents.append(
            f"{len(selected)} transactions in total, displaying only the first {numItemsLimit}"
        )
        selected = selected[:numItemsLimit]
    transListHTML.value = transListToHtmlTable(selected, colorOn=colorOn)
    totalRawAmountJPY = stats.totalRawAmountAsJPYFor(selected)
    totalAdjustedAmountJPY = stats.totalAdjustedAmountAsJPYFor(selected)
    labelStringComponents.append(
        f"{totalRawAmountJPY=:.0f}, {totalAdjustedAmountJPY=:.0f}"
    )
    transStatsLabel.value = "\n".join(labelStringComponents)

display(transStatsLabel)
display(transListHTML)

### Charts

In [ ]:
def bq_makePieFig():
    fig = bqplot.pyplot.figure(key="expByCat", animation_duration=1000)
    fig.marks = []
    pie = bqplot.pyplot.pie([], display_labels="outside", sort=False)
    figOutput = widgets.Output()

    def updateFig(options, deductIncomeOption, includeRemaining, averageByGroup):
        pieChartData = provider.pieChartData(options, deductIncomeOption, includeRemaining, averageByGroup)
        if pieChartData is None:
            with figOutput: print("No data to plot")
            pie.sizes = []
            pie.labels = []
            pie.colors = []
            return
        catToAmount = pieChartData.categoryToAmount
        totalAmount = sum(catToAmount.values())
        pairs = catToAmount.items()
        amounts = [abs(am) for _, am in pairs]
        labels = [f"{c.label} ({abs(am) / totalAmount * 100:.1f}%) {formatAmount(abs(am))} ({formatAmountInCNY(abs(am), provider)})" for c, am in pairs]
        colors = [XKCD_TO_HEX[COLORS[k.category]] for k, _ in pairs]
        opacities = [0.5 if k.isForecast else 1.0 for k, _ in pairs]
        bqplot.pyplot.figure(key="expByCat")
        total = sum(catToAmount.values())
        titlePrefix = "Avg. expenses" if averageByGroup else "Expenses"
        fig.title = f"{titlePrefix} for {options.filter.labelOption} (total: {formatAmount(total)})"
        pie.sizes = amounts
        pie.labels = labels
        pie.colors = colors
        pie.opacities = opacities
        with figOutput:
            figOutput.clear_output()
            print(pieChartData.longDescription)
    return fig, figOutput, updateFig

# bq_expensePieFig, bq_updateExpensePieFig = bq_makePieFig()
# linkWithSharedWidgets(deductIncomeOption=deductIncomeOptionWidget,
#                       includeRemaining=includeRemainingWidget)(bq_updateExpensePieFig)
# display(bq_expensePieFig)

In [ ]:
debug_out = widgets.Output()

def bq_makeBarFig():
    """Create a reusable bqplot bar fig, providing an upadteFig function."""
    fig = bqplot.pyplot.figure(key="barChart", animation_duration=1000)
    fig.marks = []

    xOrd = bq.scales.OrdinalScale()
    yLin = bq.scales.LinearScale()
    xAxis = bq.axes.Axis(scale=xOrd, tick_rotate=-30)
    yAxis = bq.axes.Axis(scale=yLin, orientation="vertical", tick_format=".2s")
    fig.axes = [xAxis, yAxis]
    incomeBars = bq.Bars(x=[], y=[], scales={'x': xOrd, 'y': yLin}, padding=0.2,
                         type="stacked", stroke="gray")
    expenseBars = bq.Bars(x=[], y=[], scales={'x': xOrd, 'y': yLin}, padding=0.6,
                          type="stacked", stroke="black")
    expenseBarTooltip = bq.Tooltip()
    expenseBars.tooltip = expenseBarTooltip

    incomeBarsTooltipOutput = widgets.Output()
    expenseBarsTooltipOutput = widgets.Output()

    def updateFig(options, deductIncomeOption):
        fig.title = f"Bar chart for {options.filter.labelOption}"
        bqplot.pyplot.figure("barChart")
        barChartData = provider.barChartData(options, deductIncomeOption)
        xOrd.domain = barChartData.labels

        incomeCats = barChartData.orderedIncomeCats
        incomeBars.x = barChartData.labels
        incomeBars.y = [[d.get(c, 0) for d in barChartData.incomeTotalsByCat] for c in incomeCats]
        incomeBars.colors = [XKCD_TO_HEX[COLORS[c.category]] for c in incomeCats]
        incomeBars.opacities = [0.1 if c.isForecast else 0.5 for c in incomeCats]

        def updateTooltip(tooltipOut, bars, hoverEvent, cats):
            data = hoverEvent["data"]
            index = data["index"]
            subIndex = data["subIndex"] if "subIndex" in data else data["sub_index"]
            try:
                category = cats[subIndex].label
                amount = bars.y[subIndex][index]
                tooltipOut.clear_output()
                with tooltipOut:
                    print(f"{category}: {formatAmount(amount)} ({formatAmountInCNY(amount, provider)})")
            except:
                pass

        def incomeBarsHovered(bars, event):
            updateTooltip(incomeBarsTooltipOutput, bars, event, incomeCats)
        def expenseBarsHovered(bars, event):
            updateTooltip(expenseBarsTooltipOutput, bars, event, expenseCats)

        incomeBars.on_hover(incomeBarsHovered)
        incomeBars.tooltip = incomeBarsTooltipOutput
        expenseBars.on_hover(expenseBarsHovered)
        expenseBars.tooltip = expenseBarsTooltipOutput

        expenseCats = barChartData.orderedExpenseCats
        expenseBars.x = barChartData.labels
        expenseBars.y = [[d[c] for d in barChartData.expenseTotalsByCat] for c in expenseCats]
        expenseBars.colors = [XKCD_TO_HEX[COLORS[c.category]] for c in expenseCats]
        expenseBars.opacities = [0.25 if c.isForecast else 1.0 for c in expenseCats]

    fig.marks = [incomeBars, expenseBars]
    return fig, incomeBars, expenseBars, updateFig


bq_pieFigForSelection, bq_pieFigForSelectionOutput, bq_updatePieFigForSelection = bq_makePieFig()
bq_barFig, bq_incomeBars, bq_expenseBars, bq_updateBarFig = bq_makeBarFig()

from dataclasses import replace

@linkWithSharedWidgets(deductIncomeOption=deductIncomeOptionWidget,
                       includeRemaining=includeRemainingWidget,
                       averageByGroup=averageByGroupWidget)
def bq_updateBarFigWithPieForSelection(options, deductIncomeOption, includeRemaining, averageByGroup):
    bq_updateBarFig(options, deductIncomeOption)

    # Handle click on bar to update pie chart.
    def barClicked(bars, event):
        barChartData = provider.barChartData(options, deductIncomeOption)
        groupLabel = barChartData.labels[event["data"]["index"]]
        providerFilterForPie = replace(options.filter, labelOption=groupLabel)
        optionsForPie = replace(options, filter=providerFilterForPie)
        bq_updatePieFigForSelection(optionsForPie, deductIncomeOption, includeRemaining, averageByGroup)

    bq_incomeBars.on_element_click(barClicked)
    bq_expenseBars.on_element_click(barClicked)

    # Reset pie fig to the whole interval.
    bq_updatePieFigForSelection(options, deductIncomeOption, includeRemaining, averageByGroup)

display(bq_barFig)
display(bq_pieFigForSelection)
display(bq_pieFigForSelectionOutput)
display(debug_out)

### Shop distribution

In [ ]:
shopDistributionOutput = widgets.Output()
@linkWithSharedWidgets(deductIncomeOption=deductIncomeOptionWidget)
def updateShopDistributionOutput(options, deductIncomeOption):
    shopDistributionOutput.clear_output()
    with shopDistributionOutput:
        for name, amount in provider.dataForShopDistribution(options, deductIncomeOption):
            print(f"{name}: {formatAmount(amount)} ({formatAmountInCNY(amount, provider)})")

display(shopDistributionOutput)